In [ ]:
import os
os.environ['USE_OFFLINE'] = 'true'
os.environ['HF_HUB_OFFLINE'] = 'true'
print('Offline mode is on; the notebook will not connect to Elasticsearch.')

# Dutch bank help search: interview walkthrough

This is the ordered, offline-first path for the 15-minute demo. Each code cell reads a small cached artifact or request definition and is designed to complete in under ten seconds.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
print(f'Working from: {ROOT}')

## 1. Setup and the offline safety net

The same environment flag controls every retrieval path. Offline mode removes the local Elasticsearch dependency while keeping BM25, vectors, and RRF available in Python.

In [ ]:
import config
print({'USE_OFFLINE': config.USE_OFFLINE, 'ES_URL': config.ES_URL, 'segment_boost': config.SEGMENT_BOOST_WEIGHT})

## 2. Fictional Dutch data and relevance labels

The corpus intentionally includes close card, account, payment, and business cases. The query set includes graded labels so search quality is measurable rather than anecdotal.

In [ ]:
pages = json.loads((ROOT / 'data' / 'pages.json').read_text(encoding='utf-8'))
queries = json.loads((ROOT / 'data' / 'queries.json').read_text(encoding='utf-8'))
print(f'{len(pages)} fictional help pages; {len(queries)} judged queries')
print(queries[0])

## 3. LLM enrichment with guardrails

A reviewed cache supplies two-sentence summaries and controlled metadata. Validation checks page coverage, allowed labels, and numeric grounding before enrichment is used for retrieval.

In [ ]:
from enrich import load_cache, validate_cache

enrichments = validate_cache(pages, load_cache())
print(f'Validated enrichment records: {len(enrichments)}')
print(enrichments[0])

## 4. Index design

The live index uses Dutch text analysis for lexical search and 384-dimensional E5 vectors for semantic search. The offline equivalent holds the same enriched documents in rank_bm25 and NumPy structures.

In [ ]:
from index import index_mapping

mapping = index_mapping()['mappings']['properties']
print({field: mapping[field] for field in ('title', 'body', 'embedding', 'audience')})

## 5. Hybrid search

The request has one keyword retriever and one nearest-neighbour retriever. Elasticsearch fuses them with reciprocal rank fusion; the Python fallback applies the same rank formula.

In [ ]:
from search import RRF_RANK_CONSTANT, RRF_RANK_WINDOW_SIZE, lexical_query

print(json.dumps(lexical_query('pas kwijt'), ensure_ascii=False, indent=2))
print({'rrf_rank_constant': RRF_RANK_CONSTANT, 'rrf_window': RRF_RANK_WINDOW_SIZE})

## 6. Explicit personalisation

A visitor chooses private, business, or no segment. The result retains its original score and its visible audience adjustment; the system does not infer a segment from a query or person.

In [ ]:
from search import apply_segment_boost

candidates = [
    {'id': 'p11', 'title': 'Betaalrekening openen', 'summary': '', 'audience': 'private', 'score': 0.020, 'method': 'hybrid'},
    {'id': 'p12', 'title': 'Zakelijke betaalrekening aanvragen', 'summary': '', 'audience': 'business', 'score': 0.021, 'method': 'hybrid'},
    {'id': 'p13', 'title': 'Extra rekening', 'summary': '', 'audience': 'both', 'score': 0.019, 'method': 'hybrid'},
]
[(item['id'], item['audience'], round(item['score'], 3), round(item['segment_boost'], 3)) for item in apply_segment_boost(candidates, 'private', 3)]

## 7. Evaluation

The saved benchmark records RR, AP, nDCG at 10, Recall at 10, query-type slices, and the weakest hybrid queries. Reading the artifact is fast; rerunning all 25 vector queries is kept outside the notebook.

In [ ]:
metrics = json.loads((ROOT / 'results' / 'metrics.json').read_text(encoding='utf-8'))
for method, values in metrics['methods'].items():
    mean = values['mean']
    print(f"{method:16} nDCG@10={mean['ndcg_at_10']:.3f}  Recall@10={mean['recall_at_10']:.3f}")
print('Worst hybrid query:', metrics['worst_hybrid_queries'][0])

## 8. Product Owner report

The report turns the saved benchmark into a portable HTML page with a method comparison, a query-type table, and recommendations in plain language.

In [ ]:
from report import write_report

report_path = write_report()
print(f'Refreshed: {report_path.name} ({report_path.stat().st_size / 1_000_000:.1f} MB)')

## 9. Governance and monitoring

The model card documents the intended use, no-go areas, summary-number guardrail, personalisation risk, privacy prompts, and concrete monitoring thresholds.

In [ ]:
model_card = (ROOT / 'docs' / 'model_card.md').read_text(encoding='utf-8')
print('\n'.join(model_card.splitlines()[:18]))

## 10. Continuous integration

Each push runs offline tests and a real 25-query relevance guard. The workflow caches the public embedding model and compiles the Java example.

In [ ]:
workflow = (ROOT / '.github' / 'workflows' / 'ci.yml').read_text(encoding='utf-8')
for phrase in ('USE_OFFLINE: "true"', 'HF_HUB_OFFLINE: "true"', 'javac java/HybridSearchExample.java'):
    print(f'{phrase}: {phrase in workflow}')

## 11. Java integration

The Java 11 example uses only the standard HTTP client. It receives the Python-created E5 vector, sends the same hybrid RRF request, and prints Elasticsearch's response.

In [ ]:
java_source = (ROOT / 'java' / 'HybridSearchExample.java').read_text(encoding='utf-8')
print('\n'.join(java_source.splitlines()[:22]))

## Live commands for the interview

Use the terminal for the deliberately slower, real retrieval commands: search.py for the three-way ranking, search.py with personalisation-demo for segment comparison, and evaluate.py only when you want to regenerate the 25-query artifact. Keep the standalone HTML report open as the Product Owner view.